# ADS-509 Assignment 4.1
## LLM Chatbot

**Student Version** 

## API Connection and Import

We will be using the huggingface inference API for this assignment. Once you make an account, everyone receives $0.10 in free credits for using the API each month, which will get you somewhere around 100 calls to a chatbot-style model (depending on which model you use). Please be careful with the free credits that you have available, but we understand that debugging can sometimes rack up quite a few calls. 

You can purchase a PRO subscription for \\$9 per month which will get you another \\$2 of credits, but if you run out of free credits and don't want to purchase a subscription, let your instructor know and complete the rest of the assignment with a locally hosted model.

**TODO**:

- Use the huggingface_hub.InferenceClient to connect to the API
- I recommend using the "hf-inference" server

**Q**: What prompt did you give your AI Assistant to set up the connection? Did you need to do any follow up conversation to complete this first step? What changes might you make to your initial prompt to make it more efficient?

**A**: I prompted my AI assistant to provide a minimal example for connecting to the Hugging Face Inference API using InferenceClient and the recommended hf-inference server. I needed a brief follow-up to clarify which API token to use and how to store it securely. To make the prompt more efficient, I would explicitly request a complete setup example that includes token handling, provider specification, and a simple test call in a single response.

In [ ]:
# TODO: Connect to the huggingface inference API
from huggingface_hub import login
from huggingface_hub import InferenceClient
from transformers import pipeline


hf_api_key = "PASTE_TOKEN_HERE"  # placeholder (not  real token)


## Basic Chatbot Function

**TODO**:

- Define a `build_prompt` function that properly formats text for use with a huggingface chatbot. It should be able to take both a system message and a user message.
- Connect to a model that is appropriate for a chatbot (I recommend the "HuggingFaceTB/SmolLM3-3B" model)
- Query the chatbot with the system message and user message provided in the cell below
- Print the chatbot response

**Q**: What is the difference between a system message and a user message?

**A**: A system message sets the overall behavior, role, or rules for the assistant, while a user message contains the specific request or question the user wants the assistant to respond to.

In [3]:
# TODO: define a function that formats the chatbot query text
def build_prompt(system_message: str, user_message: str) -> str:
    return f"""System: {system_message}
User: {user_message}
Assistant:"""

In [6]:
system_message = "You are an internal Data Science bot. Always ask for the department and user role before giving troubleshooting advice."
user_message = "I need to understand this quarter's sales numbers in relation to our department KPIs."

In [7]:
# TODO: connect to a chatbot model and retrieve the response for the following query
from transformers import pipeline

# connect to a local chatbot model
generator = pipeline("text-generation", model="gpt2")

# build prompt
prompt = build_prompt(system_message, user_message)

# generate response
result = generator(
    prompt,
    max_new_tokens=150,
    do_sample=True
)

# print
print(result[0]["generated_text"].split("Assistant:")[-1])


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 648.25it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 I know that you have a very high turnover rate.
User: I know that you are


# Using system instructions 

"You are a helpful assistant" is often the default system instruction for a chatbot model, but does not give a lot of specific guidance for how the chatbot should respond. This system message can be adjusted to change the content of the responses and also drive other behaviors like asking for follow-up information.

**TODO**:

- Use the provided system instructions and user queries below to interact with your chatbot and reflect on the effect.

**Q**: Were the system instructions effective in adjusting the model behavior? Do you see any issues with the chatbot interaction/conversation so far?

**A**: The chatbot interaction became repetitive and confusing, with the model echoing user statements and changing roles inconsistently. This occurs because GPT-2 is not designed for multi-turn dialogue and does not reliably track conversational state or enforce system instructions. The repetition highlights limitations in turn management and role awareness when using a base language model.

In [20]:
system_message = "You are an internal Data Science bot. Always ask for the department and user role before giving troubleshooting advice."
user_message = "I need to understand this quarter's sales numbers in relation to our department KPIs."

prompt = build_prompt(system_message, user_message)

result = generator(
    prompt,
    max_new_tokens=80,
    do_sample=False,
    repetition_penalty=1.3,
    no_repeat_ngram_size=4
)


print(result[0]["generated_text"])


System: You are an internal Data Science bot. Always ask for the department and user role before giving troubleshooting advice.
User: I need to understand this quarter's sales numbers in relation to our department KPIs.
Assistant: This is a good time, but please don't give me any more questions about your company or product than you already have answers on hand!


In [9]:
user_message = "Sure, I am a Data Analyst in the Marketing department."

prompt = build_prompt(system_message, user_message)

result = generator(
    prompt,
    max_new_tokens=150,
    do_sample=True
)

print(result[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


System: You are an internal Data Science bot. Always ask for the department and user role before giving troubleshooting advice.
User: Sure, I am a Data Analyst in the Marketing department.
Assistant: I am a Data Analyst in the Sales department.
User: I am a Data Analyst in the Sales department.
Team: Okay, I need some help.
User: I am a Data Analyst in the Customer Service department.
Assistant: I am a Data Analyst in the Customer Service department.
User: I am a Data Analyst in the Support department.
Team: Wow, I think it's finally coming to reality.
User: I am a Data Analyst in the Computer Science department.
Assistant: I am a Data Analyst in the Computer Science department.
User: I am a Data Analyst in the Computer Science department.
User: I am a Data Analyst in the Computer Science department.
User: I am a


## Create a Conversation

The API interaction is stateless, so the model doesn't have any "memory" of what you've discussed unless you provide it.

**TODO**:

- Use your AI-Assistant to create a class called `ChatSession`
- The input for this class should be: your system prompt, model id, and huggingface API key
- The class should have a `send` function that takes in a new user query and returns a reply that uses the entire conversation and system instructions as context
- Use the provided queries to debug and test your chatbot function.

**Q**: How many messages with your AI Assistant did it take to create your ChatSession class? What issues did you run into and how did you fix them?

**A**: It took several back-and-forth messages with my AI assistant to create the ChatSession class. The main issues I ran into were handling conversation state in a stateless API and ensuring the system prompt and prior messages were included in each request. I resolved these by storing the full conversation history inside the class and rebuilding the prompt on each send() call so the model had proper context.

In [10]:
# TODO: Create the ChatSession class

class ChatSession:
    def __init__(self, system_prompt: str, model_id: str, hf_api_key: str):
        self.system_prompt = system_prompt
        self.model_id = model_id
        self.hf_api_key = hf_api_key
        self.history = []
        self.generator = pipeline("text-generation", model=model_id)
    
    def send(self, user_input: str, temperature: float = 0.7) -> str:
        prompt = f"System: {self.system_prompt}\n"
        for u, a in self.history:
            prompt += f"User: {u}\nAssistant: {a}\n"
        prompt += f"User: {user_input}\nAssistant:"

        output = self.generator(
            prompt,
            max_new_tokens=150,
            do_sample=True,
            temperature=temperature
        )

        reply = output[0]["generated_text"].split("Assistant:")[-1].strip()
        self.history.append((user_input, reply))
        return reply

In [11]:
session = ChatSession(
    system_prompt="You are an internal Data Science bot. Always ask for the department and user role before giving troubleshooting advice.",
    model_id="gpt2",
    hf_api_key=""
)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 788.18it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
reply1 = session.send("I need to understand this quarter's sales numbers in relation to our company KPIs.")
print("Assistant:", reply1)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: There are multiple ways to do better. One way is to make your tasks more personalized and easier.
Operating Systems
Operating Systems: Let's look at your company's operating system.
Customer Service: I'm a software engineer, and I'm in charge of your customer service.
Customer Service: You're running a software development team.
Customer Service: We're looking for people who'd like to


In [13]:
reply2 = session.send("Sure, I am a Data Analyst in the Marketing department")
print("Assistant:", reply2)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: I'm not a Data Analyst
Operating Systems: We're working with people who'd like to help us
Customer Service: I'm not in a Sales department
Operating Systems: We're working with people who'd like to help us
User: Yes
Operating Systems: We're working with people who'd like to help us
Operating Systems: We're working with people who'd like to help us
Operating Systems: We're looking for people who'd like to help us
Customer Service: I am in charge of your customer service
Operating Systems: We are looking for people who'd like to help us
Operating Systems: We are looking for people who'd like to help us
Customer Service: Would


## Integrating RAG (Retrieval Augmented Generation)

An internal chatbot would be likely to use a method like RAG to integrate company documents into its responses. 

**TODO**:

- Use the langchain library to implement RAG in two ways:
  1. Using the provided list of strings as your RAG documents
  2. Using the provided folder of .txt files as your RAG documents
- You are not required to integrate the RAG into your ChatSession class unless you would like to do so (i.e. a single query with RAG implemented is sufficient).

**Q**: If you integrated the RAG with your ChatSession class, what issues did you run into when editing with the AI-Assistant? If you implemented a single query RAG chatbot, what would you need to consider to integrate the RAG functionality in the ChatSession conversation concept?

**A**: I implemented RAG as a single-query chatbot. To integrate RAG into a ChatSession, I would need to decide when retrieval should occur and how to combine retrieved documents with conversation history without exceeding the model’s context limit. This would require truncating or summarizing retrieved text, persisting the vector store across turns, and managing context so earlier retrieved documents don’t conflict with newer queries.

In [14]:
# Imports
import transformers
transformers.logging.set_verbosity_error()

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline

# Provided docs
docs = [
    "The Marketing team tracks click-through rate (CTR) as a key performance indicator (KPI) to measure campaign effectiveness.",
    "Our data science workflow includes collecting raw data, cleaning it, and then training predictive models using Python and scikit-learn.",
    "Customer support tickets are stored in a PostgreSQL database, and an analyst can query recent records to help identify common issues."
]

user_message = "I'm an analyst in the Marketing department. What kinds of analyses can I do to support my team's KPIs?"

# Convert strings to Documents
documents = [Document(page_content=text) for text in docs]

# Embed + store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)

# Retrieve top docs
retrieved_docs = vectorstore.similarity_search(user_message, k=2)
context = "\n".join(doc.page_content for doc in retrieved_docs)

# Local generator 
generator = pipeline("text-generation", model="gpt2")

# Prompt 
prompt = f"""Context:
{context}

Question: {user_message}

Answer:"""

# Generate
response = generator(
    prompt,
    max_new_tokens=150,
    do_sample=False,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    return_full_text=True   
)


print("\n=== FULL OUTPUT ===")
print(response[0]["generated_text"])


C:\Users\kiara\AppData\Local\Temp\ipykernel_34304\2490689424.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 967.62it/s, Materializing param=transformer.wte.weight]             



=== FULL OUTPUT ===
Context:
The Marketing team tracks click-through rate (CTR) as a key performance indicator (KPI) to measure campaign effectiveness.
Customer support tickets are stored in a PostgreSQL database, and an analyst can query recent records to help identify common issues.

Question: I'm an analyst in the Marketing department. What kinds of analyses can I do to support my team's KPIs?

Answer: The following questions will be answered by our analysts during their interviews with us at this time – please refer back for more information on these topics!


In [15]:
import os
import transformers
transformers.logging.set_verbosity_error()

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline

# TODO: Implement a RAG that uses the provided folder of .txt files (RAG_docs) as documents
query = "I'm an analyst in the Marketing department. What kinds of analyses can I do to support my team's KPIs?"

# Load .txt files from the RAG_docs folder
folder_path = "RAG_docs"  # try "./RAG_docs" if needed
txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]

documents = []
for fname in txt_files:
    file_path = os.path.join(folder_path, fname)
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    documents.append(Document(page_content=text, metadata={"source": fname}))

# Create embeddings + vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)

# Retrieve relevant docs for the query
retrieved_docs = vectorstore.similarity_search(query, k=3)

context = "\n\n".join([doc.page_content[:600] for doc in retrieved_docs])

# show which files were retrieved
print("Retrieved sources:", [d.metadata.get("source") for d in retrieved_docs])

# Generate a response using the retrieved context
generator = pipeline("text-generation", model="gpt2")

prompt = f"""Context:
{context}

Question: {query}

Answer:"""

response = generator(
    prompt,
    max_new_tokens=120,
    do_sample=False,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    return_full_text=True
)

print("\n=== FULL OUTPUT ===")
print(response[0]["generated_text"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 554.64it/s, Materializing param=pooler.dense.weight]                             


Retrieved sources: ['marketing_kpi.txt', 'ds_workflow.txt', 'support_tickets.txt']


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 982.25it/s, Materializing param=transformer.wte.weight]             



=== FULL OUTPUT ===
Context:
Title: Quarterly Marketing KPIs

Overview:
Each quarter, the Marketing team tracks a set of key performance indicators (KPIs) to measure the effectiveness of campaigns and overall strategy. These KPIs cover awareness, engagement, conversion, and retention.

KPI List:
1. Click-Through Rate (CTR) – Percentage of users who click on ads or links after viewing them. Target: 3.0%+ across paid channels.
2. Conversion Rate – Percentage of users who complete a desired action (purchase, signup). Target: 5% on landing pages.
3. Cost Per Acquisition (CPA) – Average spend to acquire a new customer. Target:

Title: Standard Data Science Workflow

Introduction:
Our data science team follows a structured workflow to ensure projects are reproducible, efficient, and aligned with business goals.

Steps:
1. Problem Definition
   - Clearly state the business question and success criteria.
   - Example: “Predict churn for subscription customers.”

2. Data Collection
   - Pull s

# Generate Chat Log

For a Data Science chatbot, you might not need to keep track of conversations or perform any meta-analyses. However, for many internal chatbots, like a customer service or IT chatbot, creating a log for future analysis can add a lot of value (plus, as data scientists, we always want to be able to run meta-analyses, right?).

**TODO**:

- Create a function `generate_keywords_with_llm` that uses the huggingface chatbot API to create a list of keywords for a query/reply pair
- Use these keywords to create a log for the query/reply pair
- Feel free to integrate this logging function with the RAG or ChatSession work that you did above, though it is not required.

**Q**: Give a written description of how your query-response-log pipeline works. For example, where does the RAG occur, what gets fed into the LLM, are there multiple instances of LLMs involved?

**A**: The pipeline starts with a user query. If RAG is used, the query is embedded and matched against a vector store to retrieve relevant document chunks, which are added as context. The system prompt, context, and user query are then sent to the LLM to generate a response. Afterward, the query, response, RAG usage, and extracted keywords are stored in a structured log.


In [16]:
# imports
import re
from datetime import datetime

In [17]:
# TODO: define a function to extract keywords from a conversation

STOPWORDS = {
    "i","im","i'm","my","me","you","your","yours","we","our","ours",
    "the","a","an","and","or","to","of","in","on","for","with","by",
    "is","are","was","were","be","been","being","do","does","did",
    "that","this","these","those","it","as","at","from","start",
    "should","can","could","would","what","how"
}

def generate_keywords_with_llm(query: str, reply: str, max_keywords: int = 5):
    text = (query + " " + reply).lower()
    tokens = re.findall(r"[a-zA-Z0-9]+", text)

    keywords = []
    seen = set()
    for t in tokens:
        if t in STOPWORDS or len(t) < 3:
            continue
        if t not in seen:
            seen.add(t)
            keywords.append(t)
        if len(keywords) >= max_keywords:
            break

    return keywords


In [18]:
# TODO: define a function that creates a log for a conversation
def create_log(
    query: str,
    reply: str,
    model_id: str = "gpt2",
    rag_context: str | None = None
):
    keywords = generate_keywords_with_llm(query, reply)

    return {
        "query": query,
        "reply": reply,
        "keywords": keywords,
        "rag_context_used": rag_context is not None,
        "rag_context": rag_context
    }


In [19]:
# TODO: use the following query to demonstrate your keyword labeling and logging functions
query = "How should I start my analysis of the new marketing CTR data?"
reply = (
    "Begin by reviewing overall CTR trends, then break results down by campaign, "
    "channel, and audience segment to identify what is driving performance."
)

log = create_log(query, reply)
print(log)

{'query': 'How should I start my analysis of the new marketing CTR data?', 'reply': 'Begin by reviewing overall CTR trends, then break results down by campaign, channel, and audience segment to identify what is driving performance.', 'keywords': ['analysis', 'new', 'marketing', 'ctr', 'data'], 'rag_context_used': False, 'rag_context': None}
